# TIL burden as a continuous variable (response to reviewers, Fig 5b)

**Produces** three views of overall survival against TIL⁺ spot fraction, on the 218 patients
of the survival cohort with a Visium section (TIL fraction from the frozen TIL zones):
1. **Continuous RMST curve** (`gbmcollaboration.utils.continuous_hr`): Cox model with a natural
   cubic spline of TIL fraction (3 df), turned into RMST at τ = 3 years along the TIL range,
   with its peak and inflection point. Unadjusted, adjusted for MGMT, and for MGMT + age.
2. **Spline tests** (R survival + splines): likelihood-ratio tests of the overall spline and
   of non-linearity, on the raw fraction and on log10(fraction + 0.001).
3. **Cutoff scan with survRM2**, the model of Fig 5b: RMST difference (TIL-high − TIL-low) at
   τ = 3 years for cutoffs from the 50th to the 90th percentile of TIL fraction over the 228
   sections, unadjusted and adjusted for MGMT. The 75th percentile is the Fig 5b cutoff.

**Origin**: `continuous_hr.py` and the continuous-modelling cell of
`analysis/scratch/survival/continuous_rmst.ipynb` (Jonathan Bac), which placed the inflection of
the RMST curve at 0.030, the Fig 5b cutoff (0.0304); the spline tests follow the review
reruns of November 2026.

In [ ]:
from unittest import mock

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gbmcollaboration.paper import cohort, config, output, til  # config sets R_HOME for rpy2
from gbmcollaboration.utils import continuous_hr
import rpy2.robjects as ro  # noqa: E402
from rpy2.robjects import pandas2ri  # noqa: E402
from rpy2.robjects.conversion import localconverter  # noqa: E402

ro.r("suppressPackageStartupMessages({library(survival); library(splines); library(survRM2)})")

In [ ]:
# Parameters
OUT = output.setup("fig5b_TIL_spline")
TAU = 3
TIME_COL, EVENT_COL, CELL_COL = "os_years", "os_event", "TIL_fraction"
COVARIATE_SETS = {"none": [], "MGMT": ["mgmt"], "MGMT + age": ["mgmt", "age"]}
LOG_OFFSET = 0.001
CUTOFF_QUANTILES = np.round(np.arange(0.50, 0.901, 0.05), 2)

In [ ]:
def to_r(name, df):
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv[name] = ro.conversion.py2rpy(df.reset_index(drop=True))


def spline_tests(df, x, covs):
    """LR tests of a natural cubic spline (3 df) of x in a Cox model: overall (vs no x) and
    non-linearity (vs linear x)."""
    d = df[[TIME_COL, EVENT_COL, x, *covs]].dropna()
    to_r("sd", d.rename(columns={TIME_COL: "time", EVENT_COL: "event", x: "x"}))
    adj = "".join(f" + {c}" for c in covs)
    ro.r(f"""
    f0 <- coxph(Surv(time, event) ~ 1{adj}, data = sd)
    f1 <- coxph(Surv(time, event) ~ x{adj}, data = sd)
    fs <- coxph(Surv(time, event) ~ ns(x, df = 3){adj}, data = sd)
    p_overall <- anova(f0, fs)[2, "Pr(>|Chi|)"]
    p_nonlin  <- anova(f1, fs)[2, "Pr(>|Chi|)"]
    p_linear  <- anova(f0, f1)[2, "Pr(>|Chi|)"]
    hr_linear <- exp(coef(f1)[["x"]])
    """)
    g = lambda v: float(ro.r(v)[0])  # noqa: E731
    return dict(n=len(d), p_spline_overall=g("p_overall"), p_nonlinearity=g("p_nonlin"),
                p_linear=g("p_linear"), hr_per_unit_linear=g("hr_linear"))


def rmst(df, arm, covs=(), tau=TAU):
    """survRM2::rmst2 of arm 1 vs arm 0 at tau (months), adjusted for covs if given."""
    d = df[[TIME_COL, EVENT_COL, arm, *covs]].dropna()
    d = d[d[TIME_COL] > 0]
    r = d.copy()
    r.columns = ["time", "event", "arm", *[f"c{i}" for i in range(len(covs))]]
    to_r("rd", r)
    ro.globalenv["tv"] = float(tau)
    ro.r("a <- rmst2(rd$time, rd$event, rd$arm, tau = tv)")
    res = list(ro.r("a$unadjusted.result[1,]"))
    if covs:
        cs = ", ".join(f"rd$c{i}" for i in range(len(covs)))
        res = list(ro.r(f"rmst2(rd$time, rd$event, rd$arm, tau = tv, covariates = cbind({cs}))$adjusted.result[1,]"))
    return dict(n=len(d), n_high=int(d[arm].sum()), diff_months=res[0] * 12, lo=res[1] * 12, hi=res[2] * 12, p=res[3])

## Data

TIL⁺ fraction per Visium section (frozen TIL zones, 228 sections), joined by block to the
260-patient survival cohort; MGMT and age with unknown values left missing.

In [ ]:
groups = til.til_groups()
surv = cohort.code_covariates(cohort.survival_cohort(), missing="keep")
df = groups.reset_index(names="section").merge(surv[[TIME_COL, EVENT_COL, "mgmt", "age"]],
                                                left_on="block", right_index=True)
print(f"Sections with survival data: {len(df)}; events {int(df[EVENT_COL].sum())}; "
      f"MGMT known {int(df['mgmt'].notna().sum())}; TIL fraction 0 in {int((df[CELL_COL] == 0).sum())}")
print(f"Fig 5b cutoff (Q75 over 228 sections): {groups.attrs['cutoff']:.4f}")
df["log_TIL_fraction"] = np.log10(df[CELL_COL] + LOG_OFFSET)

## 1. Continuous RMST curve

The peak is the TIL fraction with the highest modelled RMST and the inflection the point of
largest curvature; the "non-linearity p" of this module is the smallest Wald p of the
non-linear spline coefficients (the likelihood-ratio test is in section 2).

In [ ]:
curve_rows = []
for label, covs in COVARIATE_SETS.items():
    with mock.patch.object(plt, "show"):
        summary, _ = continuous_hr.master_threshold_analysis(df=df, time_col=TIME_COL, event_col=EVENT_COL,
                                                             cell_col=CELL_COL, tau=TAU, cov_names=covs,
                                                             plot_type="both")
        fig = plt.gcf()
    summary.insert(0, "covariates", label)
    curve_rows.append(summary)
    output.save(OUT, f"continuous_rmst_{label.replace(' + ', '_').replace(' ', '_')}", fig, summary,
                panel=f"Response TIL spline RMST {label}")
    plt.close(fig)
curves = pd.concat(curve_rows, ignore_index=True)
print(curves.round(4).to_string(index=False))

## 2. Spline tests

In [ ]:
tests = pd.DataFrame([dict(scale=scale, covariates=label, **spline_tests(df, x, covs))
                      for scale, x in (("raw", CELL_COL), ("log10(x + 0.001)", "log_TIL_fraction"))
                      for label, covs in COVARIATE_SETS.items()])
output.save_table(OUT, "spline_tests", tests.round(4), index=False, panel="Response TIL spline tests")
print(tests.round(4).to_string(index=False))

## 3. Cutoff scan with survRM2

Cutoffs are percentiles of TIL fraction over the 228 sections, as for Fig 5b. Nominal p
values: the scan is descriptive.

In [ ]:
scan = []
for q in CUTOFF_QUANTILES:
    cut = groups["TIL_fraction"].quantile(q)
    d = df.assign(arm=(df[CELL_COL] >= cut).astype(int))
    for label, covs in (("none", []), ("MGMT", ["mgmt"])):
        scan.append(dict(quantile=q, cutoff=cut, covariates=label, **rmst(d, "arm", covs)))
scan = pd.DataFrame(scan)
print(scan.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 4))
for (label, sub), color in zip(scan.groupby("covariates", sort=False), ("#7f7f7f", "#fe7a02")):
    ax.errorbar(sub["quantile"] + (0.004 if label == "MGMT" else -0.004), sub["diff_months"],
                yerr=[sub["diff_months"] - sub["lo"], sub["hi"] - sub["diff_months"]], fmt="o-", color=color,
                capsize=3, label=f"adjusted: {label}")
    for _, r in sub[sub["p"] < 0.05].iterrows():
        ax.annotate("*", (r["quantile"], r["hi"]), ha="center", va="bottom", color=color)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0.75, color="grey", ls="--", lw=1, label="Fig 5b cutoff (Q75)")
ax.set_xlabel("TIL-high cutoff (percentile of TIL+ fraction, 228 sections)")
ax.set_ylabel(f"ΔRMST at τ = {TAU} y, high − low (months)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
output.save(OUT, "cutoff_scan_survRM2", fig, scan, panel="Response TIL cutoff scan")

In [ ]:
print(output.versions(["numpy", "pandas", "lifelines", "patsy", "matplotlib", "rpy2"],
                      ["survival", "survRM2"]).to_string(index=False))